In Part-1, we built PD model (white box) which is fully explainable. While using Black-Box models like Neural Net, Gradient Boosting, Random Forest etc. explability is a challenge. We will use two packages LIME (Local Interpretable Model-Agnostic Explanations) and SHAP (SHapley Additive exPlanations) here to work on model explainability.

# Imported required libraries

In [1]:
# !pip install lime
# !pip install catboost
# !pip install shap

In [2]:
# For Data Handling & System Utilities
import pandas as pd
import numpy as np
import random
import os

# For Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from IPython.display import Markdown, display
from IPython.display import display, HTML

# Statistical & Distribution Analysis
from scipy.stats import gamma
import missingno as msno

# Machine Learning (Data Splitting)
from sklearn.model_selection import train_test_split

In [3]:
# Machine Learning – Data Preparation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning – Models (Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Model Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    precision_recall_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay)

In [4]:
# For exporting ML model
import pickle

In [5]:
# For statistical calculation
from scipy import stats
import math

In [6]:
# Model Explainability Libraries
import lime
import lime.lime_tabular
import shap

In [7]:
import warnings
warnings.filterwarnings("ignore")

In [8]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Importing Preprocessed Train, Test and Holdout Dataset

In [9]:
# Define the directory where the data is stored
base_path = r"C:/Users/danie/Downloads/Data Analytic/credit_loan_analysis"

try:
    # Attempt to read the specific CSV files
    # We use os.path.join for path safety
    X_train = pd.read_csv(os.path.join(base_path, 'loan_inputs_train.csv'))
    y_train = pd.read_csv(os.path.join(base_path, 'loan_targets_train.csv'))
    X_test = pd.read_csv(os.path.join(base_path, 'loan_inputs_test.csv'))
    y_test = pd.read_csv(os.path.join(base_path, 'loan_targets_test.csv'))
    X_holdout = pd.read_csv(os.path.join(base_path, 'loan_inputs_holdout.csv'))
    y_holdout = pd.read_csv(os.path.join(base_path, 'loan_targets_holdout.csv'))
    
except FileNotFoundError:
    print(f"Error: Could not find the files in '{base_path}'.")
    print("Please verify the directory path and ensure the files were exported correctly.")
except Exception as e:
    print(f"An unexpected error occurred during import: {e}")

else:
    # Success confirmation and data verification
    print("Data successfully imported.")
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_train shape: {X_test.shape}")
    print(f"y_train shape: {y_test.shape}")
    print(f"X_train shape: {X_holdout.shape}")
    print(f"y_train shape: {y_holdout.shape}")

Data successfully imported.
X_train shape: (240000, 9)
y_train shape: (240000, 1)
X_train shape: (80000, 9)
y_train shape: (80000, 1)
X_train shape: (80000, 9)
y_train shape: (80000, 1)


# Column to drop

In [10]:
# Define columns to remove
cols_to_drop = ['purpose', 'ead']

# Create a dictionary to iterate through splits and their names
splits = {'Training': X_train, 'Testing': X_test, 'Holdout': X_holdout}

print("🚀 Starting column removal...")

for name, df in splits.items():
    # Drop columns and update the dataframe in-place
    df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    
    # Verification: Check if columns still exist in the dataframe
    remaining_forbidden = [col for col in cols_to_drop if col in df.columns]
    
    if not remaining_forbidden:
        print(f"✅ {name} set: {cols_to_drop} successfully removed. (Remaining columns: {len(df.columns)})")
    else:
        print(f"❌ {name} set: Failed to remove {remaining_forbidden}")

# Final check
print(f"\nFinal Feature List (X_train): {X_train.columns.tolist()}")
print(f"\nFinal Feature List (X_test): {X_test.columns.tolist()}")
print(f"\nFinal Feature List (X_holdout): {X_holdout.columns.tolist()}")

🚀 Starting column removal...
✅ Training set: ['purpose', 'ead'] successfully removed. (Remaining columns: 7)
✅ Testing set: ['purpose', 'ead'] successfully removed. (Remaining columns: 7)
✅ Holdout set: ['purpose', 'ead'] successfully removed. (Remaining columns: 7)

Final Feature List (X_train): ['int_rate', 'interest_band', 'grade', 'loan_to_value', 'collateral', 'regularity_of_inflows', 'ltv_group']

Final Feature List (X_test): ['int_rate', 'interest_band', 'grade', 'loan_to_value', 'collateral', 'regularity_of_inflows', 'ltv_group']

Final Feature List (X_holdout): ['int_rate', 'interest_band', 'grade', 'loan_to_value', 'collateral', 'regularity_of_inflows', 'ltv_group']


In [11]:
# Combine training features (X_train) and target (y_train) into one trained dataset for reference
train_data = pd.concat([X_train, y_train], axis=1)

In [12]:
# Combine training features (X_train) and target (y_train) into one trained dataset for reference
test_data = pd.concat([X_test, y_test], axis=1)

In [13]:
# Combine training features (X_train) and target (y_train) into one trained dataset for reference
holdout_data = pd.concat([X_holdout, y_holdout], axis=1)

# Model Explainability with LIME

In [14]:
def train_credit_models_robust(X_train, y_train, X_test, y_test):
    RANDOM_STATE = 42
    
    # Separate features by type
    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X_train.select_dtypes(include=['object']).columns

    # Create a preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

    # Flatten target arrays into 1D array
    y_train_processed = y_train.values.ravel()
    y_test_processed = y_test.values.ravel()

    # Define models to train
    raw_models = [
        ("Extra Trees", ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ("XGBoost", XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, verbosity=0)),
        ("MLP Neural Net", MLPClassifier(max_iter=500, random_state=RANDOM_STATE))]
    # Store trained models and results
    trained_pipelines = []
    # Print table header
    print(f"{'Model Name':<20} | {'Train Acc':<10} | {'Test Acc':<10}")
    print("-" * 45)

    # Train each model
    for name, model in raw_models:
        try:
            # Create and train the pipeline
            clf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', model)])
            
            clf_pipeline.fit(X_train, y_train_processed)
            
            # Evaluate performance
            train_preds = clf_pipeline.predict(X_train) # Predict on training data
            test_preds = clf_pipeline.predict(X_test)   # Predict on test data
            # Calculate accuracy scores
            train_acc = accuracy_score(y_train_processed, train_preds)
            test_acc = accuracy_score(y_test_processed, test_preds)
            
            # Print results in a clean format
            print(f"{name:<20} | {train_acc:>9.4f} | {test_acc:>8.4f}")
            
            trained_pipelines.append({"name": name, "clf": clf_pipeline, "test_score": test_acc})
            
        except Exception as e:
            print(f"Failed to train {name}: {e}")

    return trained_pipelines

In [ ]:
# Execute the function
trained_list = train_credit_models_robust(X_train, y_train, X_test, y_test)

Model Name           | Train Acc  | Test Acc  
---------------------------------------------
Extra Trees          |    1.0000 |   0.9244
Random Forest        |    1.0000 |   0.9273


In [ ]:
def explain_all_models_lime(trained_list, X_train, X_test, instance_index=0, num_features=10):
    """
    Unified function to explain all models in trained_list using subplots.
    Uses optimized sampling to ensure the browser remains responsive.
    """
    # Setup Plot Grid
    num_models = len(trained_list)
    if num_models == 0:
        print("No models found in trained_list.")
        return

    # Calculate rows needed based on number of models
    cols = 2 
    rows = math.ceil(num_models / cols)
    
    # Large figure size to ensure readability
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 6))
    
    # Ensure axes is an array even if there is only 1 model
    if num_models == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    # Sample training data for LIME (performance optimization)
    X_train_sample = X_train.sample(min(1000, len(X_train)))
    
    # Prepare the specific instance to explain
    instance = X_test.iloc[[instance_index]]

    # Iterate through each model
    for idx, model_data in enumerate(trained_list):
        model_name = model_data['name']
        pipeline = model_data['clf']
        
        # Extract preprocessing and classifier parts
        preprocessor = pipeline.named_steps['preprocessor']
        classifier = pipeline.named_steps['classifier']
        
        # Reconstruct Feature Names (Unified Logic)
        numeric_cols = list(X_train.select_dtypes(include=['number']).columns)
        try:
            cat_transformer = preprocessor.named_transformers_['cat']
            encoded_cols = list(cat_transformer.get_feature_names_out())
        except:
            cat_count = X_train.select_dtypes(include=['object']).shape[1]
            encoded_cols = [f"cat_{i}" for i in range(cat_count)]
        
        feature_names = numeric_cols + encoded_cols
        
        # Transform Training Data & Instance
        train_transformed = preprocessor.transform(X_train_sample)
        if hasattr(train_transformed, "toarray"): train_transformed = train_transformed.toarray()
        
        inst_transformed = preprocessor.transform(instance)
        if hasattr(inst_transformed, "toarray"): inst_transformed = inst_transformed.toarray()
        
        # Initialize LIME Explainer
        explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=train_transformed,
            feature_names=feature_names,
            class_names=['Default', 'No Default'],
            mode='classification',
            discretize_continuous=True
        )
        
        # Generate Explanation with reduced samples for performance
        exp = explainer.explain_instance(
            data_row=inst_transformed[0], 
            predict_fn=classifier.predict_proba,
            num_features=num_features,
            num_samples=1000)
        
        # Manual Plotting to Subplot
        ax = axes[idx]
        exp_list = exp.as_list()
        
        # Reverse list to have the most important features at the top of the bar chart
        vals = [x[1] for x in exp_list][::-1]
        names = [x[0] for x in exp_list][::-1]
        
        # Assign colors:
        ## Green → increases "good" outcome
        ## Red → increases "bad" outcome
        colors = ['#27ae60' if x > 0 else '#c0392b' for x in vals]

        # Create horizontal bar chart
        ax.barh(names, vals, color=colors)
        ax.set_title(f"LIME Top {num_features}: {model_name}", fontsize=14, fontweight='bold', pad=15)
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Hide empty subplots (if grid > number of models)
    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(f"LIME Feature Importance Comparison (Instance Index: {instance_index})", fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Explain customer at index 5 across all models
explain_all_models_lime(trained_list, X_train, X_test, instance_index=5, num_features=10)

- Green Bars: Features that increase the probability of the event (pushing the prediction toward "Not Default" or a "Good" rating in this context).
- Red Bars: Features that decrease the probability (pushing the prediction toward "Default" or "High Risk" in this context).
- The Length of the Bar: Represents how much "weight" the model gave for a specific feature in this context

**Observation:** Across all five models (Extra Trees, Random Forest, Gradient Boosting, XGBoost, and MLP), there is high feature agreement regarding the primary risk drivers for this applicant.

**Primary Risk Drivers (Negative Impact)**:
- Regularity of Inflows: The lack of regular income inflows is the dominant reason for the high-risk prediction.
- Collateral Status: The absence of collateral significantly penalizes the score across all architectures.
- LTV Ratios: High Loan-to-Value metrics are consistently viewed as a negative risk factor.

**Model Behavior**
- XGBoost shows the highest sensitivity to the 'Inflow' feature.
- Tree-based models (RF, Extra Trees) show very similar "reasoning" patterns.
- MLP (Neural Net) identifies 'Grade A' status as a minor positive offset, which is less pronounced in the boosting models.

**Conclusion:** The consistency across different model types (Model Consensus) provides high confidence in the explanation for this specific credit decision.

# Checking if the above Explainability derived by LIME can be Trusted or Not

Compare local prediction by LIME vs actual prediction for all the models to verify if the above model explainabilities are reliable or not. If there is very minimal difference between local vs actual prediction, then the model explainability by LIME (above) should be trusted, else not.

In [ ]:
def verify_lime_reliability_class0(trained_list, X_train, X_test, instance_index=0):
    """
    Compares LIME's local prediction vs. actual prediction for Class 0 (Default).
    Displays a line chart and a table with Model, R2, and Default Rates.
    """
    # Create an empty list to store results for each model
    lime_metrics = []
    
    # Select ONE customer (instance) from the test set
    instance = X_test.iloc[[instance_index]]

    # Loop through each trained model
    for model_data in trained_list:
        # Extract model name and pipeline
        model_name = model_data['name']
        pipeline = model_data['clf']
        # Separate preprocessing and classifier
        preprocessor = pipeline.named_steps['preprocessor']
        classifier = pipeline.named_steps['classifier']
        
        # 1. Get Actual Prediction (Class 0: Default)
        inst_transformed = preprocessor.transform(instance)
        if hasattr(inst_transformed, "toarray"): 
            inst_transformed = inst_transformed.toarray()
            
        actual_probas = classifier.predict_proba(inst_transformed)[0]
        actual_class0_prob = actual_probas[0] 
        
        # 2. LIME Explanation & Local R2
        X_train_sample = X_train.sample(min(1000, len(X_train)))
        train_transformed = preprocessor.transform(X_train_sample)
        if hasattr(train_transformed, "toarray"): 
            train_transformed = train_transformed.toarray()

        # Initialize LIME explainer
        explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=train_transformed,
            mode='classification',
            class_names=['Default', 'No Default'])

        # Generate explanation for the selected customer
        exp = explainer.explain_instance(
            data_row=inst_transformed[0], 
            predict_fn=classifier.predict_proba,
            num_samples=1000)
        
        # Calculate Local Prob for Class 0 and extract R2 Score
        lime_local_prob_class0 = 1 - exp.local_pred[0]
        r2_score = exp.score

        # Store results in list
        lime_metrics.append({
            "Model": model_name,
            "LIME R2 Score": round(r2_score, 2),
            "Actual Default Rate": round(actual_class0_prob, 2),
            "LIME Default Rate": round(lime_local_prob_class0, 2)
        })

    # --- Visualization ---
    df = pd.DataFrame(lime_metrics)
    
    ax = df.plot(
        kind="line", 
        x="Model", 
        y=["LIME Default Rate", "Actual Default Rate"],
        marker="o", mfc="red", mec="white", markersize=12, linewidth=3,
        figsize=(20, 6), color=['#3498db', '#e67e22'])
    plt.title("Reliability Check: LIME vs. Actual Model (Default Rate / Class 0)", fontsize=20, pad=20)
    plt.ylabel("Probability", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.box(False)
    plt.show()

    # --- Display Table ---
    final_table = df[["Model", "LIME R2 Score", "Actual Default Rate", "LIME Default Rate"]]
    print("\n--- Model Reliability Table ---")
    display(final_table)
    return df

In [ ]:
# Run the function
metrics_df = verify_lime_reliability_class0(trained_list, X_train, X_test, instance_index=5)

# Model Explainability with SHAP Kernel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def explain_all_models_shap(trained_list, X_train, X_test, instance_index=0, num_features=10):
    num_models = len(trained_list)
    if num_models == 0:
        print("No models found.")
        return

    cols = 2
    rows = math.ceil(num_models / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 6))
    axes = axes.flatten() if num_models > 1 else [axes]

    # --- OPTIMIZATION START ---
    # Instead of sampling 1000 raw rows, we use kmeans to summarize the 
    # distribution into 10 representative points. This silences the warning.
    background_raw = X_train.sample(min(1000, len(X_train)), random_state=42)
    # --- OPTIMIZATION END ---

    instance = X_test.iloc[[instance_index]]

    for idx, model_data in enumerate(trained_list):
        model_name = model_data['name']
        pipeline = model_data['clf']
        preprocessor = pipeline.named_steps['preprocessor']
        classifier = pipeline.named_steps['classifier']

        # Preprocess background and instance
        bg_transformed = preprocessor.transform(background_raw)
        if hasattr(bg_transformed, "toarray"): bg_transformed = bg_transformed.toarray()
        
        inst_transformed = preprocessor.transform(instance)
        if hasattr(inst_transformed, "toarray"): inst_transformed = inst_transformed.toarray()

        # --- REFINED SUMMARY ---
        # Summarize the transformed background data into 10 clusters
        # This is what silences the red warning boxes.
        bg_summary = shap.kmeans(bg_transformed, 10)

        # Initialize Explainer with the summary
        explainer = shap.KernelExplainer(classifier.predict_proba, bg_summary)
        
        # Calculate values (nsamples=500 is usually enough for a stable local estimate)
        shap_values = explainer.shap_values(inst_transformed, nsamples=1000, l1_reg="num_features(10)")

        # Handle SHAP Output (Class 0: Default)
        if isinstance(shap_values, list):
            current_shap_vals = shap_values[0].flatten()
        else:
            current_shap_vals = shap_values[0, :, 0]

        # Reconstruct Feature Names
        numeric_cols = list(X_train.select_dtypes(include=['number']).columns)
        try:
            encoded_cols = list(preprocessor.named_transformers_['cat'].get_feature_names_out())
        except:
            encoded_cols = [f"cat_{i}" for i in range(X_train.select_dtypes(include=['object']).shape[1])]
        feature_names = numeric_cols + encoded_cols

        # Plotting logic
        indices = np.argsort(np.abs(current_shap_vals))[-num_features:]
        sorted_vals = current_shap_vals[indices]
        sorted_names = [feature_names[i] for i in indices]

        ax = axes[idx]
        colors = ['#27ae60' if x < 0 else '#c0392b' for x in sorted_vals]
        ax.barh(sorted_names, sorted_vals, color=colors)
        ax.set_title(f"SHAP: {model_name}", fontsize=14, fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.6)

    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"Optimized SHAP Explanations (Index {instance_index})", fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Execution
explain_all_models_shap(trained_list, X_train, X_test, instance_index=5)

In [ ]:
# Setup random state for consistency
RANDOM_STATE = 42

In [ ]:
def verify_shap_reliability_class0(trained_list, X_train, X_test, instance_index=0):
    """
    Verifies SHAP reliability using K-Means summarization.
    Uses the pre-defined RANDOM_STATE for sampling consistency.
    """
    shap_metrics = []
    
    # Prepare Instance
    instance = X_test.iloc[[instance_index]]
    
    for model_data in trained_list:
        model_name = model_data['name']
        pipeline = model_data['clf']
        preprocessor = pipeline.named_steps['preprocessor']
        classifier = pipeline.named_steps['classifier']
        
        # 1. Background Transformation
        # Using the existing RANDOM_STATE for consistent sampling
        background_df = X_train.sample(min(1000, len(X_train)), random_state=RANDOM_STATE)
        bg_transformed = preprocessor.transform(background_df)
        if hasattr(bg_transformed, "toarray"): 
            bg_transformed = bg_transformed.toarray()
        
        # Summarize background to 10 samples to optimize performance/silence warnings
        bg_summary = shap.kmeans(bg_transformed, 10)
        
        # 2. Instance Transformation
        inst_transformed = preprocessor.transform(instance)
        if hasattr(inst_transformed, "toarray"): 
            inst_transformed = inst_transformed.toarray()
        
        # 3. Actual Prediction (Class 0: Default)
        actual_class0_prob = classifier.predict_proba(inst_transformed)[0][0]
        
        # 4. SHAP Calculation
        explainer = shap.KernelExplainer(classifier.predict_proba, bg_summary)
        base_value = explainer.expected_value[0]
        
        # nsamples=500 provides a stable local approximation
        shap_values = explainer.shap_values(inst_transformed, nsamples=500, silent=True)
        
        # Handle SHAP multi-class vs binary output formatting
        if isinstance(shap_values, list):
            current_shap_vals = shap_values[0].flatten()
        else:
            current_shap_vals = shap_values[0, :, 0]
            
        # 5. Reliability Test: Sum of SHAP values + Base Value should = Actual Prediction
        shap_sum = current_shap_vals.sum()
        derived_pred = base_value + shap_sum
        gap = abs(actual_class0_prob - derived_pred)
        
        shap_metrics.append({
            "Model": model_name,
            "Base Value (Avg)": round(base_value, 2),
            "Sum of SHAP": round(shap_sum, 2),
            "SHAP Derived Pred": round(derived_pred, 2),
            "Actual Model Pred": round(actual_class0_prob, 2),
            "Reliability Gap": round(gap, 4)
        })

    # --- Visualization ---
    df = pd.DataFrame(shap_metrics)
    
    ax = df.plot(
        kind="line", x="Model", y=["SHAP Derived Pred", "Actual Model Pred"],
        marker="o", mfc="red", mec="white", markersize=12, linewidth=3,
        figsize=(20, 6), color=['#3498db', '#e67e22']
    )
    
    plt.title(f"SHAP Efficiency Check (Instance {instance_index})", fontsize=20, pad=20)
    plt.ylabel("Probability of Default", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.box(False)
    plt.show()

    # --- Display Table ---
    print("\n--- SHAP Reliability Metrics ---")
    display(df[["Model", "Actual Model Pred", "SHAP Derived Pred", "Reliability Gap"]])
    
    return df

# Run Verification
shap_reliability_df = verify_shap_reliability_class0(trained_list, X_train, X_test, instance_index=5)

In [ ]:
def evaluate_shap_trustworthiness(trained_list, X_train, X_test, instance_index=0):
    """
    Evaluates SHAP results based on:
    1. Efficiency Gap (Mathematical Accuracy)
    2. Faithfulness (Impact of removing top features)
    """
    results = []
    instance = X_test.iloc[[instance_index]]

    for model_data in trained_list:
        model_name = model_data['name']
        pipeline = model_data['clf']
        preprocessor = pipeline.named_steps['preprocessor']
        classifier = pipeline.named_steps['classifier']

        # 1. Setup Data
        bg_raw = X_train.sample(min(1000, len(X_train)), random_state=RANDOM_STATE)
        bg_trans = preprocessor.transform(bg_raw)
        if hasattr(bg_trans, "toarray"): bg_trans = bg_trans.toarray()
        bg_summary = shap.kmeans(bg_trans, 10)
        
        inst_trans = preprocessor.transform(instance)
        if hasattr(inst_trans, "toarray"): inst_trans = inst_trans.toarray()

        # 2. Get SHAP Values
        explainer = shap.KernelExplainer(classifier.predict_proba, bg_summary)
        base_val = explainer.expected_value[0]
        shap_vals = explainer.shap_values(inst_trans, nsamples=500, silent=True)
        
        # Format SHAP values
        current_shap = shap_vals[0].flatten() if isinstance(shap_vals, list) else shap_vals[0, :, 0]
        
        # 3. TEST A: Efficiency (Mathematical Gap)
        actual_pred = classifier.predict_proba(inst_trans)[0][0]
        derived_pred = base_val + current_shap.sum()
        efficiency_gap = abs(actual_pred - derived_pred)

        # 4. TEST B: Faithfulness (Perturbation)
        # We "neutralize" the top 3 features by replacing them with the background mean
        top_3_indices = np.argsort(np.abs(current_shap))[-3:]
        inst_perturbed = inst_trans.copy()
        
        # Replace top features with background average to see if prediction breaks
        for idx in top_3_indices:
            inst_perturbed[0, idx] = np.mean(bg_trans[:, idx])
            
        perturbed_pred = classifier.predict_proba(inst_perturbed)[0][0]
        # Prediction Change: Higher is better (means SHAP found the right triggers)
        pred_change = abs(actual_pred - perturbed_pred)

        # 5. Determine Trust Status
        status = "✅ TRUSTED" if efficiency_gap < 0.05 and pred_change > 0.01 else "⚠️ UNRELIABLE"

        results.append({
            "Model": model_name,
            "Efficiency Gap": round(efficiency_gap, 4),
            "Faithfulness (Pred Change)": round(pred_change, 4),
            "Trust Status": status
        })

    eval_df = pd.DataFrame(results)
    print(f"\n--- SHAP Reliability Evaluation (Instance {instance_index}) ---")
    display(eval_df)
    return eval_df

# Execute Evaluation
evaluation_metrics = evaluate_shap_trustworthiness(trained_list, X_train, X_test, instance_index=5)

# Saving model

In [ ]:
# Destination path
destination_path = r"C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\models"

# Create the directory if it doesn't exist yet
if not os.path.exists(destination_path):
    os.makedirs(destination_path)

print(f"--- Saving Models to: {destination_path} ---")

# Iterate through your trained_list and save each one
for model_data in trained_list:
    model_name = model_data['name'].replace(" ", "_").lower() # Clean filename
    file_name = f"{model_name}_pipeline.pkl"
    full_path = os.path.join(destination_path, file_name)
    
    try:
        with open(full_path, 'wb') as f:
            # We save the 'clf' which is the full scikit-learn Pipeline
            pickle.dump(model_data['clf'], f)
        print(f"Successfully saved: {file_name}")
    except Exception as e:
        print(f"Failed to save {model_data['name']}: {e}")

print("\nAll models have been serialized.")

In [ ]:
import shutil

# Define your source and destination
current_notebook_name = "credit_risk_PD_model (part 2).ipynb" 
destination_path = r"C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\models"

# Ensure destination exists
if not os.path.exists(destination_path):
    os.makedirs(destination_path)

# Perform the copy
try:
    # This looks for the file in the current working directory
    shutil.copy(current_notebook_name, os.path.join(destination_path, current_notebook_name))
    print(f"✅ Notebook successfully copied to: {destination_path}")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{current_notebook_name}'.")
    print("👉 Make sure the filename matches exactly and you have saved (Ctrl+S) recently.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")